# Intraday Arbitrage Backtesting — German Day-Ahead Market

> Rule-based peak/off-peak spread strategy backtested on 7+ years of SMARD data.

**Strategy:** Buy overnight (01:00–05:00), sell during business hours (09:00–19:00), weekdays only.  
**Data:** SMARD API — hourly German day-ahead prices (2018–present).  
**Cost assumption:** 7.0 €/MWh (transmission ~3.5 + imbalance ~2.5 + transaction ~1.0)

---

## 1 — Data Collection

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

print('Libraries loaded.')

# ── Fetch all available hourly blocks from SMARD API ────────
INDEX_URL = 'https://www.smard.de/app/chart_data/4169/DE/index_hour.json'
timestamps = requests.get(INDEX_URL).json()['timestamps']
blocks     = sorted(timestamps)

records = []
for i, ts in enumerate(blocks):
    url  = f'https://www.smard.de/app/chart_data/4169/DE/4169_DE_hour_{ts}.json'
    data = requests.get(url).json().get('series', [])
    for entry in data:
        if entry[1] is not None:
            records.append({
                'timestamp': pd.to_datetime(entry[0], unit='ms', utc=True),
                'price':     entry[1]   # EUR/MWh — no conversion needed
            })
    if i % 20 == 0:
        print(f'  {i+1}/{len(blocks)} blocks fetched...')

# ── Build DataFrame ─────────────────────────────────────────
df = (pd.DataFrame(records)
        .drop_duplicates('timestamp')
        .sort_values('timestamp')
        .set_index('timestamp')
        .tz_convert('Europe/Berlin'))

df = df[df['price'] > -500]   # remove extreme outliers
df['hour']    = df.index.hour
df['weekday'] = df.index.dayofweek

print(f'\n{len(df):,} hourly records loaded')
print(f'Period : {df.index.min().date()} → {df.index.max().date()}')

## 2 — Peak / Off-Peak Analysis

In [ ]:
# ── Define peak / off-peak windows ──────────────────────────
# EPEX convention: weekday 08:00–20:00 = peak
df['is_weekend'] = df['weekday'] >= 5
df['is_peak']    = (~df['is_weekend']) & df['hour'].between(8, 19)
df['year']       = df.index.year

# ── Annual peak vs off-peak summary ─────────────────────────
summary = df.groupby(['year', 'is_peak'])['price'].mean().unstack()
summary.columns = ['Off-Peak', 'Peak']
summary['Spread'] = (summary['Peak'] - summary['Off-Peak']).round(1)
print('Annual Peak vs Off-Peak Prices (€/MWh):')
print(summary.round(1))

# ── Visualise ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Spread by year
summary['Spread'].plot(kind='bar', ax=axes[0], color='#378ADD', alpha=0.85)
axes[0].axhline(0, color='red', linewidth=0.8, linestyle='--')
axes[0].set_title('Peak/Off-Peak Spread by Year (€/MWh)')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Spread (€/MWh)')
axes[0].tick_params(axis='x', rotation=45)

# Average price by hour of day
hourly = df.groupby('hour')['price'].mean()
colors = ['#E24B4A' if 8 <= h <= 19 else '#378ADD' for h in hourly.index]
axes[1].bar(hourly.index, hourly.values, color=colors, alpha=0.85)
axes[1].set_title('Average Price by Hour (red=peak, blue=off-peak)')
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Average Price (€/MWh)')
axes[1].set_xticks(range(0, 24, 2))

plt.tight_layout()
plt.savefig('peak_offpeak_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved.')

## 3 — Rule-Based Backtesting Engine

In [ ]:
# ── Strategy parameters ──────────────────────────────────────
BUY_HOURS  = (1, 5)    # buy window  : 01:00 – 05:00 (overnight off-peak)
SELL_HOURS = (9, 19)   # sell window : 09:00 – 19:00 (business hours peak)
COST_EUR_MWH = 7.0     # realistic round-trip cost (transmission + imbalance + transaction)

df_bt = df.copy()
df_bt['date'] = df_bt.index.date

results = []

for date, day_data in df_bt.groupby('date'):
    if len(day_data) < 20:
        continue

    if pd.Timestamp(date).dayofweek >= 5:   # weekdays only
        continue

    buy_window  = day_data[day_data['hour'].between(*BUY_HOURS)]
    sell_window = day_data[day_data['hour'].between(*SELL_HOURS)]

    if buy_window.empty or sell_window.empty:
        continue

    avg_buy   = buy_window['price'].mean()
    avg_sell  = sell_window['price'].mean()
    gross_pnl = avg_sell - avg_buy
    net_pnl   = gross_pnl - COST_EUR_MWH

    results.append({
        'date':       date,
        'avg_buy':    round(avg_buy, 2),
        'avg_sell':   round(avg_sell, 2),
        'gross_pnl':  round(gross_pnl, 2),
        'net_pnl':    round(net_pnl, 2),
        'profitable': net_pnl > 0
    })

bt = pd.DataFrame(results)
bt['date']    = pd.to_datetime(bt['date'])
bt['cum_pnl'] = bt['net_pnl'].cumsum()
bt['year']    = bt['date'].dt.year

print(f'{len(bt)} trading days backtested')
print(f"\nAnnual average net P&L (€/MWh/day):")
print(bt.groupby('year')['net_pnl'].mean().round(2))

## 4 — Risk Metrics

In [ ]:
# ── Performance metrics ──────────────────────────────────────
daily_returns = bt['net_pnl']

# Sharpe ratio (annualised, risk-free rate = 0)
sharpe = (daily_returns.mean() / daily_returns.std()) * np.sqrt(252)

# Maximum drawdown
cumulative   = bt['cum_pnl']
rolling_max  = cumulative.cummax()
drawdown     = cumulative - rolling_max
max_drawdown = drawdown.min()
max_dd_pct   = (drawdown / rolling_max.replace(0, np.nan)).min() * 100

# Profit factor
gross_profit  = daily_returns[daily_returns > 0].sum()
gross_loss    = daily_returns[daily_returns < 0].abs().sum()
profit_factor = gross_profit / gross_loss if gross_loss > 0 else np.inf

print('=' * 45)
print('  BACKTEST RESULTS — Peak/Off-Peak Strategy')
print('=' * 45)
print(f'  Period          : 2018-10-01 → 2026-05-08')
print(f'  Trading days    : {len(bt)}')
print(f'  Win rate        : {bt["profitable"].mean()*100:.1f}%')
print(f'  Avg daily P&L   : {daily_returns.mean():.2f} €/MWh')
print(f'  Sharpe ratio    : {sharpe:.2f}')
print(f'  Max drawdown    : {max_drawdown:.1f} €/MWh ({max_dd_pct:.1f}%)')
print(f'  Profit factor   : {profit_factor:.2f}')
print(f'  Total P&L       : {cumulative.iloc[-1]:.0f} €/MWh')
print('=' * 45)

# ── Equity curve + drawdown chart ───────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(bt['date'], bt['cum_pnl'], color='#378ADD', linewidth=1.5)
axes[0].fill_between(bt['date'], bt['cum_pnl'], alpha=0.15, color='#378ADD')
axes[0].axhline(0, color='red', linewidth=0.8, linestyle='--')
axes[0].set_title('Cumulative P&L — Peak/Off-Peak Strategy (€/MWh)')
axes[0].set_ylabel('Cumulative P&L (€/MWh)')

axes[1].fill_between(bt['date'], drawdown, 0, color='#E24B4A', alpha=0.6)
axes[1].set_title('Drawdown (€/MWh)')
axes[1].set_ylabel('Drawdown (€/MWh)')
axes[1].set_xlabel('Date')

plt.tight_layout()
plt.savefig('backtest_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved.')

## 5 — Annual Performance Summary

In [ ]:
# ── Annual breakdown ─────────────────────────────────────────
annual = bt.groupby('year').agg(
    avg_pnl    = ('net_pnl', 'mean'),
    win_rate   = ('profitable', 'mean'),
    total_days = ('net_pnl', 'count')
).round(2)
annual['win_rate'] = (annual['win_rate'] * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Annual P&L
colors = ['#378ADD' if x > 0 else '#E24B4A' for x in annual['avg_pnl']]
axes[0].bar(annual.index, annual['avg_pnl'], color=colors, alpha=0.85)
axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[0].set_title('Average Daily Net P&L by Year (€/MWh)')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Avg Daily P&L (€/MWh)')
axes[0].tick_params(axis='x', rotation=45)
for i, (yr, row) in enumerate(annual.iterrows()):
    axes[0].text(i, row['avg_pnl'] + (0.5 if row['avg_pnl'] >= 0 else -1.5),
                 f"{row['avg_pnl']:.1f}", ha='center', fontsize=9)

# Win rate
axes[1].plot(annual.index, annual['win_rate'],
             marker='o', color='#1D9E75', linewidth=2)
axes[1].axhline(50, color='red', linewidth=0.8,
                linestyle='--', label='50% breakeven')
axes[1].fill_between(annual.index, annual['win_rate'], 50,
                     where=annual['win_rate'] >= 50,
                     alpha=0.2, color='#1D9E75')
axes[1].fill_between(annual.index, annual['win_rate'], 50,
                     where=annual['win_rate'] < 50,
                     alpha=0.2, color='#E24B4A')
axes[1].set_title('Win Rate by Year (%)')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Win Rate (%)')
axes[1].set_ylim(0, 100)
axes[1].legend()
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('annual_performance.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Key findings ─────────────────────────────────────────────
print('Key findings for CV / LinkedIn:')
print(f'  → Win rate        : {bt["profitable"].mean()*100:.1f}%')
print(f'  → Avg daily P&L   : {daily_returns.mean():.2f} €/MWh')
print(f'  → Profit factor   : {profit_factor:.2f}x')
print(f'  → Max drawdown    : {max_drawdown:.0f} €/MWh')
print(f'  → Best year       : 2022 (49.9 €/MWh/day — energy crisis)')
print(f'  → Strategy decay  : 2025-2026 spread compressed by renewables')